# Lab 13 (Week 3 — Lab 3): Convolutional Neural Networks (CNNs)

Welcome to Lab 13! In this lab you will build your first Convolutional Neural Network (CNN) for image classification, using filters, pooling, and dropout.

**For detailed theory on convolutions, pooling, and CNN architecture, see the Lab 13 PDF and Learning Resources.**

This notebook focuses on the practical, hands-on workflow.

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.
- Compare your output with the **Expected Output** shown below each exercise where provided.
- Training a CNN can take a few minutes per run — be patient, especially on the full dataset.

## Getting the Dataset

This lab uses Fashion-MNIST, built directly into Keras — no download or upload needed.

In [ ]:
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)

In [ ]:
# Look at a few sample images
plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## Preparing the Data

Normalize pixel values to 0-1, and add a channel dimension since Keras expects one even for grayscale images.

In [ ]:
X_train_prepared = X_train.astype("float32") / 255.0
X_test_prepared = X_test.astype("float32") / 255.0

X_train_prepared = X_train_prepared.reshape(-1, 28, 28, 1)
X_test_prepared = X_test_prepared.reshape(-1, 28, 28, 1)

print("Prepared training shape:", X_train_prepared.shape)

## Example 1: Building a CNN

In [ ]:
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

model.summary()

## Example 2: Compiling & Training

To keep training time reasonable in this lab, we'll train on a subset of the data first.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train_prepared[:10000], y_train[:10000],
    epochs=8,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

## Example 3: Visualizing Training Progress

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss over Training")

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy over Training")

plt.tight_layout()
plt.show()

## Example 4: Evaluating and Viewing Predictions

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_prepared, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")

predictions = model.predict(X_test_prepared[:8], verbose=0)
predicted_labels = predictions.argmax(axis=1)

plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    correct = predicted_labels[i] == y_test[i]
    color = "green" if correct else "red"
    plt.title(f"{class_names[predicted_labels[i]]}", color=color, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Practice Exercise 1: Adding a Third Convolutional Block

**Exercise:** Rebuild the model from Example 1, but add a third Conv2D + MaxPooling2D block (e.g., 128 filters) before the Flatten layer. Train it on the same 10,000-image subset for 8 epochs and compare test accuracy to Example 1.

In [ ]:
model_deeper = None
test_accuracy_deeper = None

### YOUR CODE HERE ###
model_deeper = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])
model_deeper.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_deeper.fit(X_train_prepared[:10000], y_train[:10000], epochs=8, batch_size=64,
                 validation_split=0.2, verbose=0)
_, test_accuracy_deeper = model_deeper.evaluate(X_test_prepared, y_test, verbose=0)
### END ###

print("Test accuracy (deeper CNN):", test_accuracy_deeper)
print("Test accuracy (Example 1 CNN):", test_accuracy)

## Practice Exercise 2: Changing the Dropout Rate

**Exercise:** Rebuild the original architecture from Example 1, but change the Dropout rate from 0.3 to 0.5. Train and compare test accuracy and the training/validation gap to the original.

In [ ]:
### YOUR CODE HERE ###
model_d5 = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")
])
model_d5.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_d5 = model_d5.fit(X_train_prepared[:10000], y_train[:10000], epochs=8, batch_size=64,
                          validation_split=0.2, verbose=0)
_, acc_d5 = model_d5.evaluate(X_test_prepared, y_test, verbose=0)

print("Test accuracy (dropout 0.5):", acc_d5)
print("Test accuracy (dropout 0.3):", test_accuracy)
print("Train-val gap (0.5):", history_d5.history["accuracy"][-1] - history_d5.history["val_accuracy"][-1])
print("Train-val gap (0.3):", history.history["accuracy"][-1] - history.history["val_accuracy"][-1])
### END ###

## Practice Exercise 3: Viewing Misclassified Images

**Exercise:** Using your trained model from Example 1, find at least 4 images in the test set that were misclassified (predicted label != true label), and display them with both the predicted and true class name in the title.

In [ ]:
### YOUR CODE HERE ###
pred_all = model.predict(X_test_prepared, verbose=0).argmax(axis=1)
wrong = np.where(pred_all != y_test)[0][:8]

plt.figure(figsize=(12, 6))
for i, idx in enumerate(wrong):
    plt.subplot(2, 4, i + 1)
    plt.imshow(X_test[idx], cmap="gray")
    plt.title(f"Pred: {class_names[pred_all[idx]]}\nTrue: {class_names[y_test[idx]]}", fontsize=9, color="red")
    plt.axis("off")
plt.tight_layout()
plt.show()
### END ###

## Lab Tasks

Complete the following in this notebook, below this cell.

1. **Build & Train Your Own CNN**: Design your own CNN architecture (choose number of Conv2D/MaxPooling2D blocks, filter counts, and dropout rate). Train it on at least 10,000 training images and report test accuracy.
2. **Filter Count Comparison**: Train two versions of the same architecture — one with fewer filters (e.g., 16, 32) and one with more filters (e.g., 64, 128) in the Conv2D layers. Compare test accuracy and training time between them.
3. **Training Curves & Overfitting**: Plot the training/validation loss and accuracy curves for your best model. In a markdown cell, state whether you observe overfitting, and what evidence supports your answer.
4. **Per-Class Performance**: Using `sklearn.metrics.classification_report`, report precision, recall, and F1-score for each of the 10 clothing categories on the test set. Which category does your model struggle with most?
5. **Reflection**: In a markdown cell, write 3-4 sentences on how a CNN's approach to processing images (filters, pooling) differs from the Dense network you used in Lab 12, and why that difference matters for image data.

In [ ]:
# Task 1
import time

def build_cnn(f1, f2, dropout=0.3):
    m = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(f1, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(f2, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(10, activation="softmax")
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

my_cnn = build_cnn(32, 64, 0.3)
my_history = my_cnn.fit(X_train_prepared[:10000], y_train[:10000], epochs=8, batch_size=64,
                        validation_split=0.2, verbose=0)
_, my_acc = my_cnn.evaluate(X_test_prepared, y_test, verbose=0)
print(f"Task 1 test accuracy: {my_acc:.4f}")

In [ ]:
# Task 2
results, models, histories = {}, {}, {}
for name, (f1, f2) in {"small (16, 32)": (16, 32), "large (64, 128)": (64, 128)}.items():
    models[name] = build_cnn(f1, f2)
    start = time.time()
    histories[name] = models[name].fit(X_train_prepared[:10000], y_train[:10000], epochs=8,
                                       batch_size=64, validation_split=0.2, verbose=0)
    elapsed = time.time() - start
    _, acc = models[name].evaluate(X_test_prepared, y_test, verbose=0)
    results[name] = (acc, elapsed)

print("Filters          | Test Accuracy | Training Time (s)")
for name, (acc, t) in results.items():
    print(f"{name:16} | {acc:.4f}        | {t:.1f}")
best_name = max(results, key=lambda k: results[k][0])
print("Best:", best_name)

In [ ]:
# Task 3
best_model = models[best_name]
best_history = histories[best_name]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(best_history.history["loss"], label="Training Loss")
plt.plot(best_history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("Loss")
plt.subplot(1, 2, 2)
plt.plot(best_history.history["accuracy"], label="Training Accuracy")
plt.plot(best_history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.title("Accuracy")
plt.tight_layout()
plt.show()

**Task 3 – Overfitting:** Training accuracy keeps rising across epochs while validation accuracy levels off, and validation loss stops improving (or begins to rise) while training loss keeps falling. This growing gap between training and validation performance is the evidence of mild overfitting. Dropout keeps it from getting severe. (Confirm this against your own plot and adjust the wording if your curves look different.)

In [ ]:
# Task 4
from sklearn.metrics import classification_report

y_pred = best_model.predict(X_test_prepared, verbose=0).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=class_names))

**Task 4 – Weakest category:** The model struggles most with **Shirt**, which has the lowest precision, recall and F1-score. Shirts look very similar to T-shirt/top, Pullover and Coat, so they are often confused with them. (Confirm this against your own report.)

**Task 5 – Reflection:** The Dense network in Lab 12 treats its input as a flat list of numbers, so flattening an image throws away which pixels are neighbours. A CNN instead slides small filters across the image, which lets it detect local patterns like edges and textures wherever they appear, using far fewer weights because each filter is shared across the whole image. Pooling then shrinks the feature maps and makes the model less sensitive to small shifts in position. This matters for image data because meaning lives in spatial structure, and stacking conv layers builds from simple edges up to whole shapes such as a shoe or a coat.